In [ ]:
using Pkg
Pkg.activate("..")
using Revise

In [ ]:
using bslLD, Plots, Statistics
bslLD.greet()

In [ ]:
grid =  bslLD.Grid([-30.0,-4.0],[30.0,4.0],[64,65],0.1,500,1)

# initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi)

f = bslLD.Distribution(grid, 0.01,initFuncv=initFuncv);
e = bslLD.empty_vectorfield(grid);


In [ ]:
function step(f,grid)
    bslLD.advectX!(f,grid)
    e.data[1] .=- 0.1 .* sin.(2*pi .* grid.xaxes[1] ./ (grid.max[1]-grid.min[1]))
    bslLD.advectV!(f,grid,e)
end

function stepSelfConsitent(f,grid)
    bslLD.advectX!(f,grid)
    rho = bslLD.compute_density(f,grid)
    phi = -1*bslLD.adiabatic(rho,grid)
    e = bslLD.compute_e(phi,grid)
    bslLD.advectV!(f,grid,e)


end

In [ ]:
rhodiag = []
fdiag = []


for i in grid.itime
    stepSelfConsitent(f,grid)
    if i%1==0
        push!(fdiag,f.data[:,:])
        push!(rhodiag, bslLD.compute_density(f,grid).data[:])
    end
end


In [ ]:
num_frames = size(fdiag, 1)
frames_to_plot = 1:num_frames

animation = @animate for i in frames_to_plot
    heatmap(transpose(fdiag[i]),
        title = "Frame $i",
        xlabel = "x",
        ylabel = "v",
    )

end
gif(animation, "fdiag_heatmap_animation.gif", fps = 10)

In [ ]:
plot(rhodiag[10])

In [ ]:
plot(map(x->(mean((x.-mean(x)).^2)), rhodiag), yscale=:log)